In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Show up to 2 decimal places and no scientific notation
pd.options.display.float_format = '{:.2f}'.format

In [2]:
df = pd.read_csv("final_dataset_new.csv")

In [3]:
df.head()

,Country,Year,Population,Land_Use(ha),Water_Use(m3),Animals_Slaughtered,CH4_Emissions,N2O_Emissions,CO2_Emissions
0,Afghanistan,1961,9214082,37750000.00,25850000000.00,12656000.00,25.23,0.93,960.29
1,Afghanistan,1962,9404410,37800000.00,25850000000.00,13136393.00,25.73,0.92,971.12
2,Afghanistan,1963,9604491,37850000.00,25850000000.00,13606952.00,26.84,0.94,1009.19
3,Afghanistan,1964,9814317,37905000.00,25850000000.00,14188000.00,27.19,0.96,1022.71
4,Afghanistan,1965,10036003,37910000.00,25850000000.00,14986000.00,27.89,0.98,1048.98


In [4]:
# Store the original dataframe in case I modify anything
df_backup = df.copy()

In [5]:
# Convert text to numerical as our model will work better with numerical data
# This will be a new column called 'Country_Code'
label_encoder = LabelEncoder()
df['Country_Code'] = label_encoder.fit_transform(df['Country'])

# Confirm changes
df.dtypes

Country                 object
Year                     int64
Population               int64
Land_Use(ha)           float64
Water_Use(m3)          float64
Animals_Slaughtered    float64
CH4_Emissions          float64
N2O_Emissions          float64
CO2_Emissions          float64
Country_Code             int32
dtype: object

In [6]:
# Change the order of the columns
new_order = [
    'Country', 
    'Country_Code',
    'Year', 
    'Population', 
    'Animals_Slaughtered', 
    'Land_Use(ha)', 
    'Water_Use(m3)', 
    'CH4_Emissions', 
    'N2O_Emissions', 
    'CO2_Emissions'
]

# Apply the new order
df = df[new_order]
df.head()

,Country,Country_Code,Year,Population,Animals_Slaughtered,Land_Use(ha),Water_Use(m3),CH4_Emissions,N2O_Emissions,CO2_Emissions
0,Afghanistan,0,1961,9214082,12656000.00,37750000.00,25850000000.00,25.23,0.93,960.29
1,Afghanistan,0,1962,9404410,13136393.00,37800000.00,25850000000.00,25.73,0.92,971.12
2,Afghanistan,0,1963,9604491,13606952.00,37850000.00,25850000000.00,26.84,0.94,1009.19
3,Afghanistan,0,1964,9814317,14188000.00,37905000.00,25850000000.00,27.19,0.96,1022.71
4,Afghanistan,0,1965,10036003,14986000.00,37910000.00,25850000000.00,27.89,0.98,1048.98


In [11]:
# Drop any empty values in CO2_Emissions column (this was causing a NaN error)
df = df.dropna(subset=['CO2_Emissions'])

In [13]:
# Defining the features (X) and target (y) variables
X = df[['Population', 'Animals_Slaughtered', 'Land_Use(ha)', 'Water_Use(m3)']]
y = df['CO2_Emissions']

In [15]:
# This function splits the data into training and testing data
# test_size tells the model to train itself using 80% of the data and then perform testing with the other 20% (0.2)
# random_state ensures that the split is never done differently as this would skew results
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [17]:
# Preprocessing steps: impute missing values, then scale
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

In [19]:
# Train and evaluate
def evaluate(name, pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    
    mae  = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2   = r2_score(y_test, predictions)
    
    print(f"Model : {name}")
    print(f"MAE   : {mae:.2f} kt")
    print(f"RMSE  : {rmse:.2f} kt")
    print(f"R²    : {r2:.4f}")
    
    return predictions

In [24]:
rf_pred = evaluate('Random Forest', rf_pipeline, X_train, y_train, X_test, y_test)

Model : Random Forest
MAE   : 47.16 kt
RMSE  : 134.54 kt
R²    : 0.9984


In [25]:
# Creating the Random Forest model
rf = RandomForestRegressor(n_estimators = 100, random_state = 42)
rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [34]:
# Testing My Model
# Get the most recent stats
recent_stats = df[df['Year'] == df['Year'].max()].copy()

# Predict the future evaluations by reducing meat and associated resource use
# We assume reducing animals by 50% also saves 30% of water and 20% of land
future_stats = recent_stats.copy()
future_stats['Animals_Slaughtered'] = future_stats['Animals_Slaughtered'] * 0.50
future_stats['Land_Use(ha)'] = future_stats['Land_Use(ha)'] * 0.80
future_stats['Water_Use(m3)'] = future_stats['Water_Use(m3)'] * 0.70

# Predict the future evaluations
X_future = future_stats[['Population', 'Animals_Slaughtered', 'Land_Use(ha)', 'Water_Use(m3)']]
future_stats['Simulated_CO2'] = rf.predict(X_future)

# Compare the most recent stats and the future predictions
total_recent = recent_stats['CO2_Emissions'].sum()
total_future = future_stats['Simulated_CO2'].sum()
reduction = ((total_recent - total_future) / total_recent) * 100

print(f"Recent Global Emissions: {total_recent:,.0f} kt")
print(f"Simulated (50% less meat) Emissions: {total_future:,.0f} kt")
print(f"Total Percentage Reduction: {reduction:.2f}%")

Current Global Emissions: 286,569 kt
Simulated (50% less meat) Emissions: 213,187 kt
Total Percentage Reduction: 25.61%
